# Lab 1 — Distance Metrics

**Day 04 · Distance-Based ML & MLOps · Cisco AI/ML Training**

---

## Goals

1. Represent two loans as **numeric vectors** in feature space.
2. Compute **Euclidean**, **Manhattan**, and **cosine** distance/similarity.
3. Explain why **scaling** matters before distance-based algorithms (preview Lab 2 KNN).
4. Compare how each metric reacts when you change the loan pair.

> **Quick check:** cosine similarity ≈ **0.90** · Euclidean ≈ **69370** · Manhattan ≈ **71367**

**Data:** `data/lending-club/lending_club_sample.csv` (**1000** rows)

## Why this matters

KNN (Lab 2) classifies a loan by finding its **nearest neighbors** in feature space. The distance formula you choose — and whether features are **scaled** — changes who counts as "close." Day 2 NumPy norms and dot products reappear here on real Lending Club vectors.

## Three ways to measure "closeness"

| Metric | Formula (vectors a, b) | Intuition |
|--------|------------------------|----------|
| **Euclidean** | $\sqrt{\sum (a_i - b_i)^2}$ | Straight-line distance |
| **Manhattan** | $\sum \|a_i - b_i\|$ | Grid / city-block distance |
| **Cosine similarity** | $\frac{a \cdot b}{\|a\|\|b\|}$ | Angle between directions (scale-free) |

Cosine **distance** = $1 - \text{similarity}$. Values near **1** similarity mean the vectors point in similar directions even if magnitudes differ.

## Linear algebra refresher

Each loan is a **vector** in ℝ⁵. Distance metrics use **norms** and **dot products**:

| Operation | NumPy | Role in this lab |
|-----------|-------|------------------|
| L2 norm | `np.linalg.norm(v)` | Euclidean distance |
| L1 norm | `np.abs(v).sum()` | Manhattan distance |
| Dot product | `np.dot(a, b)` | Cosine similarity numerator |

## Distance metric cheat sheet

| Task | Code |
|------|------|
| Euclidean | `np.sqrt(np.sum((a - b) ** 2))` |
| Manhattan | `np.sum(np.abs(a - b))` |
| Cosine sim | `np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))` |
| Difference vector | `a - b` |

---

## 0. Locate the Lending Club file

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

LENDING_CLUB_CSV = GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv"
DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]

print("GH root:", GH_ROOT)
print("Lending Club CSV exists:", LENDING_CLUB_CSV.is_file())
print("Path:", LENDING_CLUB_CSV.relative_to(GH_ROOT))


GH root: C:\25-Trainings\2-Confirmed\10-June-26-AI-ML-Cisco\GH
Lending Club CSV exists: True
Path: data\lending-club\lending_club_sample.csv


### 0b. Load **1000** loans and derive `default` label

In [2]:
df = pd.read_csv(LENDING_CLUB_CSV)
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

print(f"rows: {len(df)}")
print(f"columns: {len(df.columns)}")
print(f"default rate: {df['default'].mean():.4f}")
display(df[NUMERIC_FEATURES + ["default"]].head(3))


rows: 1000
columns: 10
default rate: 0.4850


,loan_amnt,int_rate,annual_inc,dti,installment,default
0,34498,14.50,35829,15.66,98.07,1
1,33265,6.75,105184,5.91,859.60,0
2,4012,6.48,139128,13.90,320.56,1


### 0c. Feature ranges — why scale matters

In [3]:
ranges = df[NUMERIC_FEATURES].agg(["min", "max", "mean"]).round(2)
display(ranges)
print("annual_inc spans orders of magnitude vs int_rate — distance without scaling is dominated by income.")


,loan_amnt,int_rate,annual_inc,dti,installment
min,1040.00,5.03,25107.00,5.03,51.15
max,34999.00,23.99,179778.00,35.00,899.85
mean,18117.78,14.31,103132.76,19.71,463.44


annual_inc spans orders of magnitude vs int_rate — distance without scaling is dominated by income.


### 0d. `describe()` on numeric features

In [4]:
display(df[NUMERIC_FEATURES].describe().round(2))


,loan_amnt,int_rate,annual_inc,dti,installment
count,1000.00,1000.00,1000.00,1000.00,1000.00
mean,18117.78,14.31,103132.76,19.71,463.44
std,9968.01,5.49,45192.41,8.47,244.77
min,1040.00,5.03,25107.00,5.03,51.15
25%,9173.50,9.52,63004.00,12.44,243.05
50%,18290.00,14.35,102884.00,19.30,451.08
75%,26676.25,19.20,141821.50,27.01,682.26
max,34999.00,23.99,179778.00,35.00,899.85


---

## 1. Pick two loan feature vectors (rows 0 and 1)

In [5]:
sample = df[NUMERIC_FEATURES].iloc[:2].to_numpy(dtype=float)
a, b = sample[0], sample[1]

print(f"point A (first loan):  {a.round(2)}")
print(f"point B (second loan): {b.round(2)}")
print(f"vector shapes: a={a.shape}, b={b.shape}")


point A (first loan):  [3.4498e+04 1.4500e+01 3.5829e+04 1.5660e+01 9.8070e+01]
point B (second loan): [3.32650e+04 6.75000e+00 1.05184e+05 5.91000e+00 8.59600e+02]
vector shapes: a=(5,), b=(5,)


### 1b. Difference vector `a - b`

In [6]:
diff = a - b
print("difference vector:", diff.round(2))
print("L2 norm of diff (Euclidean):", round(float(np.linalg.norm(diff)), 4))
print("L1 norm of diff (Manhattan):", round(float(np.abs(diff).sum()), 4))


difference vector: [ 1.2330e+03  7.7500e+00 -6.9355e+04  9.7500e+00 -7.6153e+02]
L2 norm of diff (Euclidean): 69370.1405
L1 norm of diff (Manhattan): 71367.03


### 1c. Per-feature absolute gaps

In [7]:
gaps = pd.DataFrame({
    "feature": NUMERIC_FEATURES,
    "loan_a": a.round(2),
    "loan_b": b.round(2),
    "abs_diff": np.abs(a - b).round(2),
}).sort_values("abs_diff", ascending=False)
display(gaps)


,feature,loan_a,loan_b,abs_diff
2,annual_inc,35829.00,105184.00,69355.00
0,loan_amnt,34498.00,33265.00,1233.00
4,installment,98.07,859.60,761.53
3,dti,15.66,5.91,9.75
1,int_rate,14.50,6.75,7.75


---

## 2. Compute distances (checkpoint pair)

In [8]:
euclidean = float(np.sqrt(np.sum((a - b) ** 2)))
manhattan = float(np.sum(np.abs(a - b)))
cosine_similarity = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
cosine_distance = 1.0 - cosine_similarity

print(f"Euclidean distance: {euclidean:.4f}")
print(f"Manhattan distance: {manhattan:.4f}")
print(f"cosine similarity: {cosine_similarity:.4f}")
print(f"cosine distance: {cosine_distance:.4f}")


Euclidean distance: 69370.1405
Manhattan distance: 71367.0300
cosine similarity: 0.8960
cosine distance: 0.1040


### Reading the numbers

Euclidean and Manhattan distances are **large** because `annual_inc` and `installment` are on very different scales than `int_rate` and `dti`. Cosine similarity is high (~**0.90**) because the vectors share similar **direction** — a hint that cosine can be useful when magnitude matters less than pattern.

### 2b. Dot product breakdown for cosine

In [9]:
dot_ab = float(np.dot(a, b))
norm_a = float(np.linalg.norm(a))
norm_b = float(np.linalg.norm(b))
print(f"a·b = {dot_ab:.2f}")
print(f"||a|| = {norm_a:.2f}, ||b|| = {norm_b:.2f}")
print(f"cosine = dot / (||a|| * ||b||) = {dot_ab / (norm_a * norm_b):.4f}")


a·b = 4916297997.40
||a|| = 49737.71, ||b|| = 110322.13
cosine = dot / (||a|| * ||b||) = 0.8960


### 2c. Manhattan vs Euclidean ratio

In [10]:
ratio = manhattan / euclidean
print(f"Manhattan / Euclidean ratio: {ratio:.4f}")
print("In high dimensions Manhattan often grows faster relative to Euclidean (curse of dimensionality preview).")


Manhattan / Euclidean ratio: 1.0288
In high dimensions Manhattan often grows faster relative to Euclidean (curse of dimensionality preview).


---

## 3. Per-feature contribution to Manhattan distance

In [11]:
contrib = pd.DataFrame(
    {
        "feature": NUMERIC_FEATURES,
        "abs_diff": np.abs(a - b).round(2),
        "pct_of_manhattan": (np.abs(a - b) / manhattan * 100).round(1),
    }
).sort_values("abs_diff", ascending=False)
display(contrib)


,feature,abs_diff,pct_of_manhattan
2,annual_inc,69355.00,97.2
0,loan_amnt,1233.00,1.7
4,installment,761.53,1.1
3,dti,9.75,0.0
1,int_rate,7.75,0.0


### 3b. Which feature dominates?

In [12]:
top_feature = contrib.iloc[0]["feature"]
print(f"Largest Manhattan contributor: {top_feature}")
print(f"Share of total Manhattan distance: {contrib.iloc[0]['pct_of_manhattan']:.1f}%")


Largest Manhattan contributor: annual_inc
Share of total Manhattan distance: 97.2%


---

## 4. Extension — try rows 10 and 50 (`iloc[9]`, `iloc[49]`)

In [13]:
a2 = df[NUMERIC_FEATURES].iloc[9].to_numpy(dtype=float)
b2 = df[NUMERIC_FEATURES].iloc[49].to_numpy(dtype=float)

euclidean2 = float(np.sqrt(np.sum((a2 - b2) ** 2)))
manhattan2 = float(np.sum(np.abs(a2 - b2)))
cosine_sim2 = float(np.dot(a2, b2) / (np.linalg.norm(a2) * np.linalg.norm(b2)))

print(f"pair (rows 10 & 50) — Euclidean: {euclidean2:.2f}, cosine: {cosine_sim2:.4f}")


pair (rows 10 & 50) — Euclidean: 26730.40, cosine: 0.9994


In [14]:
compare = pd.DataFrame(
    {
        "pair": ["rows 0 & 1", "rows 10 & 50"],
        "euclidean": [euclidean, euclidean2],
        "manhattan": [manhattan, manhattan2],
        "cosine_sim": [cosine_similarity, cosine_sim2],
    }
)
display(compare.round(4))


,pair,euclidean,manhattan,cosine_sim
0,rows 0 & 1,69370.1405,71367.03,0.8960
1,rows 10 & 50,26730.4008,31759.32,0.9994


### Which metric changed most?

Compare the **relative** change in each column between the two pairs. Cosine similarity often shifts less than raw Euclidean when income scales differ.

In [15]:
pct_change = compare.set_index("pair").pct_change().iloc[1] * 100
print("Percent change (rows 10&50 vs rows 0&1):")
for metric, val in pct_change.items():
    print(f"  {metric}: {val:+.1f}%")


Percent change (rows 10&50 vs rows 0&1):
  euclidean: -61.5%
  manhattan: -55.5%
  cosine_sim: +11.5%


---

## 5. Why scaling matters (preview for Lab 2)

Without scaling, KNN treats a $10,000 difference in `annual_inc` as far more important than a 5-point `int_rate` gap. Lab 2 wraps `StandardScaler` in the pipeline for this reason.

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(df[NUMERIC_FEATURES].iloc[:2])
sa, sb = scaled[0], scaled[1]

euclidean_scaled = float(np.sqrt(np.sum((sa - sb) ** 2)))
cosine_scaled = float(np.dot(sa, sb) / (np.linalg.norm(sa) * np.linalg.norm(sb)))

print(f"Euclidean (scaled): {euclidean_scaled:.4f}")
print(f"cosine similarity (scaled): {cosine_scaled:.4f}")


Euclidean (scaled): 4.4721
cosine similarity (scaled): -1.0000


### 5b. Unscaled vs scaled distance comparison

In [17]:
scale_compare = pd.DataFrame({
    "metric": ["Euclidean", "cosine_sim"],
    "unscaled": [euclidean, cosine_similarity],
    "scaled_pair": [euclidean_scaled, cosine_scaled],
}).round(4)
display(scale_compare)


,metric,unscaled,scaled_pair
0,Euclidean,69370.1405,4.4721
1,cosine_sim,0.8960,-1.0000


### 5c. Z-score one feature manually (`int_rate`)

In [18]:
ir = df["int_rate"].to_numpy(dtype=float)
z0 = (ir[0] - ir.mean()) / ir.std()
z1 = (ir[1] - ir.mean()) / ir.std()
print(f"int_rate z-scores for loans 0 and 1: {z0:.3f}, {z1:.3f}")
print(f"abs z-gap: {abs(z0 - z1):.3f}")


int_rate z-scores for loans 0 and 1: 0.034, -1.378
abs z-gap: 1.412


---

## 6. Geometry — unit vectors and angle

In [19]:
ua, ub = a / np.linalg.norm(a), b / np.linalg.norm(b)
angle_rad = np.arccos(np.clip(np.dot(ua, ub), -1.0, 1.0))
angle_deg = np.degrees(angle_rad)
print(f"angle between loans 0 and 1: {angle_deg:.1f} degrees")
print(f"cosine similarity matches cos(angle): {cosine_similarity:.4f}")


angle between loans 0 and 1: 26.4 degrees
cosine similarity matches cos(angle): 0.8960


### 6b. Unit vectors preserve direction only

In [20]:
print("unit vector a:", np.round(ua, 4))
print("unit vector b:", np.round(ub, 4))
print("||ua||:", round(float(np.linalg.norm(ua)), 6))


unit vector a: [6.936e-01 3.000e-04 7.204e-01 3.000e-04 2.000e-03]
unit vector b: [3.015e-01 1.000e-04 9.534e-01 1.000e-04 7.800e-03]
||ua||: 1.0


---

## 7. Mini distance matrix (first 5 loans)

In [21]:
block = df[NUMERIC_FEATURES].iloc[:5].to_numpy(dtype=float)
n = len(block)
dist_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        dist_matrix[i, j] = np.linalg.norm(block[i] - block[j])

labels = [f"loan_{i}" for i in range(n)]
display(pd.DataFrame(dist_matrix.round(1), index=labels, columns=labels))


,loan_0,loan_1,loan_2,loan_3,loan_4
loan_0,0.0,69370.1,107703.9,31181.4,67085.9
loan_1,69370.1,0.0,44813.2,57819.6,12999.4
loan_2,107703.9,44813.2,0.0,86276.3,41204.1
loan_3,31181.4,57819.6,86276.3,0.0,50086.6
loan_4,67085.9,12999.4,41204.1,50086.6,0.0


### 7b. Nearest neighbor to loan 0 (Euclidean, unscaled)

In [22]:
dists_from_0 = dist_matrix[0, 1:]
nearest_idx = int(np.argmin(dists_from_0)) + 1
print(f"Nearest to loan 0 (among loans 1-4): loan_{nearest_idx}, distance={dists_from_0.min():.1f}")


Nearest to loan 0 (among loans 1-4): loan_3, distance=31181.4


---

## 8. When to use which metric

| Scenario | Prefer |
|----------|--------|
| Features on similar scales | Euclidean |
| High-dimensional sparse text | Cosine |
| Grid-like movement / robust to outliers | Manhattan |
| Mixed-scale tabular (this lab) | **Scale first**, then Euclidean (Lab 2) |

---

## 9. Try it yourself

1. Compute cosine similarity between loans at `iloc[0]` and `iloc[99]`.
2. Which pair is more similar — (0,1) or (0,99) by cosine?
3. Does Euclidean agree?

In [23]:
# Your code here (optional — solution in next cell)


In [24]:
c = df[NUMERIC_FEATURES].iloc[99].to_numpy(dtype=float)
cos_0_99 = float(np.dot(a, c) / (np.linalg.norm(a) * np.linalg.norm(c)))
euc_0_99 = float(np.linalg.norm(a - c))
print(f"cosine(0,99): {cos_0_99:.4f} vs cosine(0,1): {cosine_similarity:.4f}")
print(f"Euclidean(0,99): {euc_0_99:.2f} vs Euclidean(0,1): {euclidean:.2f}")


cosine(0,99): 0.7597 vs cosine(0,1): 0.8960


Euclidean(0,99): 118527.84 vs Euclidean(0,1): 69370.14


### 9b. Pairwise cosine similarities (first 4 loans)

In [25]:
block4 = df[NUMERIC_FEATURES].iloc[:4].to_numpy(dtype=float)
cos_pairs = []
for i in range(4):
    for j in range(i + 1, 4):
        vi, vj = block4[i], block4[j]
        sim = float(np.dot(vi, vj) / (np.linalg.norm(vi) * np.linalg.norm(vj)))
        cos_pairs.append({"loan_i": i, "loan_j": j, "cosine_sim": round(sim, 4)})
display(pd.DataFrame(cos_pairs))


,loan_i,loan_j,cosine_sim
0,0,1,0.8960
1,0,2,0.7401
2,0,3,0.8206
3,1,2,0.9617
4,1,3,0.9890
5,2,3,0.9916


### 9c. Feature correlation heatmap preview

In [26]:
corr = df[NUMERIC_FEATURES].corr().round(3)
display(corr)


,loan_amnt,int_rate,annual_inc,dti,installment
loan_amnt,1.000,0.033,-0.053,0.002,0.009
int_rate,0.033,1.000,-0.008,-0.012,-0.014
annual_inc,-0.053,-0.008,1.000,0.042,-0.012
dti,0.002,-0.012,0.042,1.000,-0.048
installment,0.009,-0.014,-0.012,-0.048,1.000


### 9d. StandardScaler on full column (preview Lab 2)

In [27]:
from sklearn.preprocessing import StandardScaler
full_scaled = StandardScaler().fit_transform(df[NUMERIC_FEATURES])
print("scaled matrix shape:", full_scaled.shape)
print("column means after scale:", np.round(full_scaled.mean(axis=0), 4))


scaled matrix shape: (1000, 5)
column means after scale: [ 0.  0.  0. -0. -0.]


---

## 10. Checkpoint summary

In [28]:
assert abs(cosine_similarity - 0.8960) < 0.01
assert abs(euclidean - 69370.1405) < 100
assert abs(manhattan - 71367.0300) < 100
print("Numbers match — you're good.")



Numbers match — you're good.


---

## Reflection

1. Which feature contributed most to Manhattan distance for the first pair?
2. When would cosine similarity be preferred over Euclidean distance?
3. How does scaling change the story before applying KNN?
